# El Yazısı Rakam Tanıma - 5 CNN Modeli ile ESP32-CAM için

Bu notebook 5 farklı CNN mimarisini (SqueezeNet, EfficientNet, ResNet, MobileNet, ShuffleNet) MNIST veri seti üzerinde eğitir ve ESP32-CAM'de çalıştırılmak üzere TensorFlow Lite formatına dönüştürür.

**Önemli:** ESP32-CAM'in bellek sınırlamaları nedeniyle tüm modeller küçük ve optimize edilmiştir.

In [ ]:
# Gerekli kütüphaneleri yükle
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

## 1. MNIST Veri Setini Yükle ve Hazırla

In [ ]:
# MNIST veri setini yükle
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

# Normalize et (0-1 aralığına)
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Kanal boyutu ekle (28, 28) -> (28, 28, 1)
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

# One-hot encoding
y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)

print(f"Training data shape: {x_train.shape}")
print(f"Test data shape: {x_test.shape}")
print(f"Number of classes: 10")

In [ ]:
# Örnek görüntüleri göster
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i].squeeze(), cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Yardımcı Fonksiyonlar

In [ ]:
# Eğitim callback'leri
def get_callbacks():
    return [
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
    ]

# Model eğit ve değerlendir
def train_and_evaluate(model, model_name, epochs=30, batch_size=128):
    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history = model.fit(
        x_train, y_train_cat,
        batch_size=batch_size,
        epochs=epochs,
        validation_split=0.1,
        callbacks=get_callbacks(),
        verbose=1
    )
    
    # Test değerlendirmesi
    test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
    print(f"\n{model_name} Test Accuracy: {test_acc*100:.2f}%")
    
    return history, test_acc

## 3. Model Tanımları

ESP32-CAM için optimize edilmiş 5 CNN mimarisi:

### 3.1 SqueezeNet-Micro
Fire modülleri ile parametre verimli model

In [ ]:
def fire_module(x, squeeze_filters, expand_filters, name):
    """Fire module: squeeze (1x1) + expand (1x1 + 3x3)"""
    # Squeeze layer
    squeeze = layers.Conv2D(squeeze_filters, (1, 1), activation='relu', 
                            padding='same', name=f'{name}_squeeze')(x)
    
    # Expand layers
    expand_1x1 = layers.Conv2D(expand_filters, (1, 1), activation='relu', 
                               padding='same', name=f'{name}_expand_1x1')(squeeze)
    expand_3x3 = layers.Conv2D(expand_filters, (3, 3), activation='relu', 
                               padding='same', name=f'{name}_expand_3x3')(squeeze)
    
    # Concatenate
    return layers.Concatenate(name=f'{name}_concat')([expand_1x1, expand_3x3])

def create_squeezenet(input_shape=(28, 28, 1), num_classes=10):
    """SqueezeNet-Micro for MNIST"""
    inputs = layers.Input(shape=input_shape, name='input')
    
    # Initial convolution
    x = layers.Conv2D(32, (3, 3), strides=1, activation='relu', padding='same', name='conv1')(inputs)
    x = layers.MaxPooling2D((2, 2), name='pool1')(x)
    
    # Fire modules
    x = fire_module(x, squeeze_filters=8, expand_filters=16, name='fire1')
    x = fire_module(x, squeeze_filters=8, expand_filters=16, name='fire2')
    x = layers.MaxPooling2D((2, 2), name='pool2')(x)
    
    x = fire_module(x, squeeze_filters=16, expand_filters=32, name='fire3')
    x = fire_module(x, squeeze_filters=16, expand_filters=32, name='fire4')
    
    # Final layers
    x = layers.Dropout(0.3)(x)
    x = layers.Conv2D(num_classes, (1, 1), activation='relu', name='conv_final')(x)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    outputs = layers.Activation('softmax', name='softmax')(x)
    
    model = Model(inputs, outputs, name='SqueezeNet_Micro')
    return model

squeezenet = create_squeezenet()
squeezenet.summary()

### 3.2 EfficientNet-Micro
Compound scaling ile optimize edilmiş model

In [ ]:
def mbconv_block(x, in_filters, out_filters, expand_ratio, kernel_size, stride, name):
    """Mobile Inverted Bottleneck Convolution (MBConv) block"""
    
    # Expansion phase
    expanded_filters = in_filters * expand_ratio
    if expand_ratio != 1:
        x = layers.Conv2D(expanded_filters, (1, 1), padding='same', use_bias=False, 
                          name=f'{name}_expand_conv')(x)
        x = layers.BatchNormalization(name=f'{name}_expand_bn')(x)
        x = layers.ReLU(6., name=f'{name}_expand_relu')(x)
    
    # Depthwise convolution
    x = layers.DepthwiseConv2D(kernel_size, strides=stride, padding='same', 
                                use_bias=False, name=f'{name}_dw_conv')(x)
    x = layers.BatchNormalization(name=f'{name}_dw_bn')(x)
    x = layers.ReLU(6., name=f'{name}_dw_relu')(x)
    
    # Squeeze and Excitation (simplified)
    se = layers.GlobalAveragePooling2D(name=f'{name}_se_gap')(x)
    se = layers.Reshape((1, 1, expanded_filters), name=f'{name}_se_reshape')(se)
    se = layers.Conv2D(expanded_filters // 4, (1, 1), activation='relu', 
                       padding='same', name=f'{name}_se_reduce')(se)
    se = layers.Conv2D(expanded_filters, (1, 1), activation='sigmoid', 
                       padding='same', name=f'{name}_se_expand')(se)
    x = layers.Multiply(name=f'{name}_se_multiply')([x, se])
    
    # Output phase
    x = layers.Conv2D(out_filters, (1, 1), padding='same', use_bias=False, 
                      name=f'{name}_project_conv')(x)
    x = layers.BatchNormalization(name=f'{name}_project_bn')(x)
    
    return x

def create_efficientnet(input_shape=(28, 28, 1), num_classes=10):
    """EfficientNet-Micro for MNIST"""
    inputs = layers.Input(shape=input_shape, name='input')
    
    # Stem
    x = layers.Conv2D(16, (3, 3), strides=1, padding='same', use_bias=False, name='stem_conv')(inputs)
    x = layers.BatchNormalization(name='stem_bn')(x)
    x = layers.ReLU(6., name='stem_relu')(x)
    
    # MBConv blocks
    x = mbconv_block(x, 16, 16, expand_ratio=1, kernel_size=3, stride=1, name='block1')
    x = mbconv_block(x, 16, 24, expand_ratio=6, kernel_size=3, stride=2, name='block2')
    x = mbconv_block(x, 24, 24, expand_ratio=6, kernel_size=3, stride=1, name='block3')
    x = mbconv_block(x, 24, 40, expand_ratio=6, kernel_size=5, stride=2, name='block4')
    x = mbconv_block(x, 40, 40, expand_ratio=6, kernel_size=5, stride=1, name='block5')
    
    # Head
    x = layers.Conv2D(64, (1, 1), padding='same', use_bias=False, name='head_conv')(x)
    x = layers.BatchNormalization(name='head_bn')(x)
    x = layers.ReLU(6., name='head_relu')(x)
    
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    
    model = Model(inputs, outputs, name='EfficientNet_Micro')
    return model

efficientnet = create_efficientnet()
efficientnet.summary()

### 3.3 ResNet-Micro
Residual bağlantılar ile derin ağ

In [ ]:
def residual_block(x, filters, stride=1, name='res'):
    """Residual block with skip connection"""
    shortcut = x
    
    # First convolution
    x = layers.Conv2D(filters, (3, 3), strides=stride, padding='same', 
                      use_bias=False, name=f'{name}_conv1')(x)
    x = layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = layers.ReLU(name=f'{name}_relu1')(x)
    
    # Second convolution
    x = layers.Conv2D(filters, (3, 3), strides=1, padding='same', 
                      use_bias=False, name=f'{name}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name}_bn2')(x)
    
    # Skip connection
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, (1, 1), strides=stride, padding='same',
                                 use_bias=False, name=f'{name}_shortcut_conv')(shortcut)
        shortcut = layers.BatchNormalization(name=f'{name}_shortcut_bn')(shortcut)
    
    x = layers.Add(name=f'{name}_add')([x, shortcut])
    x = layers.ReLU(name=f'{name}_relu2')(x)
    
    return x

def create_resnet(input_shape=(28, 28, 1), num_classes=10):
    """ResNet-Micro for MNIST"""
    inputs = layers.Input(shape=input_shape, name='input')
    
    # Stem
    x = layers.Conv2D(16, (3, 3), strides=1, padding='same', use_bias=False, name='stem_conv')(inputs)
    x = layers.BatchNormalization(name='stem_bn')(x)
    x = layers.ReLU(name='stem_relu')(x)
    
    # Residual blocks
    x = residual_block(x, 16, stride=1, name='res1a')
    x = residual_block(x, 16, stride=1, name='res1b')
    
    x = residual_block(x, 32, stride=2, name='res2a')
    x = residual_block(x, 32, stride=1, name='res2b')
    
    x = residual_block(x, 64, stride=2, name='res3a')
    x = residual_block(x, 64, stride=1, name='res3b')
    
    # Head
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    
    model = Model(inputs, outputs, name='ResNet_Micro')
    return model

resnet = create_resnet()
resnet.summary()

### 3.4 MobileNet-Micro
Depthwise separable convolutions ile verimli model

In [ ]:
def depthwise_separable_block(x, filters, stride=1, name='ds'):
    """Depthwise separable convolution block"""
    # Depthwise convolution
    x = layers.DepthwiseConv2D((3, 3), strides=stride, padding='same', 
                                use_bias=False, name=f'{name}_dw_conv')(x)
    x = layers.BatchNormalization(name=f'{name}_dw_bn')(x)
    x = layers.ReLU(6., name=f'{name}_dw_relu')(x)
    
    # Pointwise convolution
    x = layers.Conv2D(filters, (1, 1), strides=1, padding='same', 
                      use_bias=False, name=f'{name}_pw_conv')(x)
    x = layers.BatchNormalization(name=f'{name}_pw_bn')(x)
    x = layers.ReLU(6., name=f'{name}_pw_relu')(x)
    
    return x

def create_mobilenet(input_shape=(28, 28, 1), num_classes=10):
    """MobileNet-Micro for MNIST"""
    inputs = layers.Input(shape=input_shape, name='input')
    
    # Stem
    x = layers.Conv2D(16, (3, 3), strides=1, padding='same', use_bias=False, name='stem_conv')(inputs)
    x = layers.BatchNormalization(name='stem_bn')(x)
    x = layers.ReLU(6., name='stem_relu')(x)
    
    # Depthwise separable blocks
    x = depthwise_separable_block(x, 32, stride=1, name='ds1')
    x = depthwise_separable_block(x, 32, stride=2, name='ds2')
    
    x = depthwise_separable_block(x, 64, stride=1, name='ds3')
    x = depthwise_separable_block(x, 64, stride=2, name='ds4')
    
    x = depthwise_separable_block(x, 128, stride=1, name='ds5')
    x = depthwise_separable_block(x, 128, stride=1, name='ds6')
    
    # Head
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    
    model = Model(inputs, outputs, name='MobileNet_Micro')
    return model

mobilenet = create_mobilenet()
mobilenet.summary()

### 3.5 ShuffleNet-Micro
Channel shuffle ile verimli grup konvolüsyonları

In [ ]:
def channel_shuffle(x, groups, name='shuffle'):
    """Channel shuffle operation"""
    batch_size, height, width, channels = x.shape
    channels_per_group = channels // groups
    
    # Reshape to (batch, h, w, groups, channels_per_group)
    x = layers.Reshape((height, width, groups, channels_per_group), name=f'{name}_reshape1')(x)
    # Permute to (batch, h, w, channels_per_group, groups)
    x = layers.Permute((1, 2, 4, 3), name=f'{name}_permute')(x)
    # Reshape back to (batch, h, w, channels)
    x = layers.Reshape((height, width, channels), name=f'{name}_reshape2')(x)
    
    return x

def shuffle_unit(x, out_channels, stride, groups=2, name='shuffle_unit'):
    """ShuffleNet unit with channel shuffle"""
    in_channels = x.shape[-1]
    bottleneck_channels = out_channels // 4
    
    # Shortcut
    if stride == 2:
        shortcut = layers.AveragePooling2D((3, 3), strides=2, padding='same', 
                                           name=f'{name}_shortcut_pool')(x)
    else:
        shortcut = x
    
    # 1x1 group convolution
    x = layers.Conv2D(bottleneck_channels, (1, 1), padding='same', 
                      use_bias=False, name=f'{name}_gconv1')(x)
    x = layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = layers.ReLU(name=f'{name}_relu1')(x)
    
    # Channel shuffle
    x = channel_shuffle(x, groups, name=f'{name}_shuffle')
    
    # 3x3 depthwise convolution
    x = layers.DepthwiseConv2D((3, 3), strides=stride, padding='same', 
                                use_bias=False, name=f'{name}_dw_conv')(x)
    x = layers.BatchNormalization(name=f'{name}_bn2')(x)
    
    # 1x1 group convolution
    out_ch = out_channels - in_channels if stride == 2 else out_channels
    x = layers.Conv2D(out_ch, (1, 1), padding='same', 
                      use_bias=False, name=f'{name}_gconv2')(x)
    x = layers.BatchNormalization(name=f'{name}_bn3')(x)
    
    # Combine
    if stride == 2:
        x = layers.Concatenate(name=f'{name}_concat')([shortcut, x])
    else:
        x = layers.Add(name=f'{name}_add')([shortcut, x])
    
    x = layers.ReLU(name=f'{name}_relu2')(x)
    return x

def create_shufflenet(input_shape=(28, 28, 1), num_classes=10):
    """ShuffleNet-Micro for MNIST"""
    inputs = layers.Input(shape=input_shape, name='input')
    
    # Stem
    x = layers.Conv2D(24, (3, 3), strides=1, padding='same', use_bias=False, name='stem_conv')(inputs)
    x = layers.BatchNormalization(name='stem_bn')(x)
    x = layers.ReLU(name='stem_relu')(x)
    
    # Stage 2
    x = shuffle_unit(x, out_channels=48, stride=2, name='stage2_unit1')
    x = shuffle_unit(x, out_channels=48, stride=1, name='stage2_unit2')
    
    # Stage 3
    x = shuffle_unit(x, out_channels=96, stride=2, name='stage3_unit1')
    x = shuffle_unit(x, out_channels=96, stride=1, name='stage3_unit2')
    
    # Head
    x = layers.Conv2D(128, (1, 1), padding='same', use_bias=False, name='head_conv')(x)
    x = layers.BatchNormalization(name='head_bn')(x)
    x = layers.ReLU(name='head_relu')(x)
    
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    
    model = Model(inputs, outputs, name='ShuffleNet_Micro')
    return model

shufflenet = create_shufflenet()
shufflenet.summary()

## 4. Modelleri Eğit

In [ ]:
# Tüm modelleri sakla
models = {
    'SqueezeNet': create_squeezenet(),
    'EfficientNet': create_efficientnet(),
    'ResNet': create_resnet(),
    'MobileNet': create_mobilenet(),
    'ShuffleNet': create_shufflenet()
}

histories = {}
accuracies = {}

# Her modeli eğit
for name, model in models.items():
    history, acc = train_and_evaluate(model, name, epochs=30)
    histories[name] = history
    accuracies[name] = acc

In [ ]:
# Sonuçları karşılaştır
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)
for name, acc in accuracies.items():
    print(f"{name:15s} - Test Accuracy: {acc*100:.2f}%")

In [ ]:
# Eğitim grafiklerini göster
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for idx, (name, history) in enumerate(histories.items()):
    ax = axes.flat[idx]
    ax.plot(history.history['accuracy'], label='Training')
    ax.plot(history.history['val_accuracy'], label='Validation')
    ax.set_title(f'{name} Accuracy')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()

# Hide the empty subplot
axes.flat[-1].axis('off')

plt.tight_layout()
plt.show()

## 5. TensorFlow Lite Dönüşümü (INT8 Quantization)

In [ ]:
# Dizinleri oluştur
os.makedirs('models', exist_ok=True)
os.makedirs('model_headers', exist_ok=True)

def representative_dataset():
    """Quantization için temsili veri seti"""
    for i in range(500):
        yield [x_train[i:i+1].astype(np.float32)]

In [ ]:
def convert_to_tflite(model, model_name):
    """Model'i TFLite formatına dönüştür (INT8 quantization)"""
    
    # Converter oluştur
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    
    # INT8 quantization ayarları
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    converter.representative_dataset = representative_dataset
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.int8
    converter.inference_output_type = tf.int8
    
    # Dönüştür
    tflite_model = converter.convert()
    
    # Kaydet
    tflite_path = f'models/{model_name.lower()}_digit.tflite'
    with open(tflite_path, 'wb') as f:
        f.write(tflite_model)
    
    model_size = len(tflite_model) / 1024
    print(f"{model_name}: {model_size:.2f} KB")
    
    return tflite_model, tflite_path

# Tüm modelleri dönüştür
print("Converting models to TFLite (INT8 quantized):")
print("="*50)

tflite_models = {}
for name, model in models.items():
    tflite_model, path = convert_to_tflite(model, name)
    tflite_models[name] = (tflite_model, path)

## 6. TFLite Modellerini Doğrula

In [ ]:
def evaluate_tflite_model(tflite_path, x_test, y_test, num_samples=1000):
    """TFLite modelinin doğruluğunu test et"""
    
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    # Quantization parametreleri
    input_scale, input_zero_point = input_details[0]['quantization']
    output_scale, output_zero_point = output_details[0]['quantization']
    
    correct = 0
    for i in range(min(num_samples, len(x_test))):
        # Input'u quantize et
        input_data = x_test[i:i+1]
        if input_scale != 0:
            input_data = (input_data / input_scale + input_zero_point).astype(np.int8)
        else:
            input_data = input_data.astype(np.float32)
        
        interpreter.set_tensor(input_details[0]['index'], input_data)
        interpreter.invoke()
        
        output_data = interpreter.get_tensor(output_details[0]['index'])
        predicted = np.argmax(output_data)
        
        if predicted == y_test[i]:
            correct += 1
    
    accuracy = correct / min(num_samples, len(x_test))
    return accuracy

# Tüm TFLite modellerini doğrula
print("\nTFLite Model Validation (INT8):")
print("="*50)

for name, (_, path) in tflite_models.items():
    accuracy = evaluate_tflite_model(path, x_test, y_test)
    print(f"{name}: {accuracy*100:.2f}%")

## 7. C Header Dosyaları Oluştur (ESP32 için)

In [ ]:
def convert_to_c_array(tflite_model, model_name):
    """TFLite modelini C array formatına dönüştür"""
    
    c_array_name = f"{model_name.lower()}_model"
    
    # C header dosyası oluştur
    header_content = f'''// Auto-generated TFLite model for ESP32-CAM
// Model: {model_name}
// Size: {len(tflite_model)} bytes

#ifndef {model_name.upper()}_MODEL_H
#define {model_name.upper()}_MODEL_H

#include <stdint.h>

const unsigned int {c_array_name}_len = {len(tflite_model)};

alignas(8) const unsigned char {c_array_name}[] = {{
'''
    
    # Binary veriyi hex formatına dönüştür
    hex_values = []
    for i, byte in enumerate(tflite_model):
        hex_values.append(f'0x{byte:02x}')
    
    # Her satırda 12 değer
    lines = []
    for i in range(0, len(hex_values), 12):
        line = ', '.join(hex_values[i:i+12])
        lines.append(f'  {line}')
    
    header_content += ',\n'.join(lines)
    header_content += f'''\n}};

#endif // {model_name.upper()}_MODEL_H
'''
    
    # Dosyaya kaydet
    header_path = f'model_headers/{model_name.lower()}_model.h'
    with open(header_path, 'w') as f:
        f.write(header_content)
    
    return header_path

# Tüm modelleri C array formatına dönüştür
print("\nGenerating C header files:")
print("="*50)

for name, (tflite_model, _) in tflite_models.items():
    header_path = convert_to_c_array(tflite_model, name)
    print(f"{name}: {header_path}")

## 8. Model Özeti ve Karşılaştırma

In [ ]:
import pandas as pd

# Model bilgilerini topla
summary_data = []

for name in models.keys():
    model = models[name]
    tflite_model, tflite_path = tflite_models[name]
    
    # Model boyutu
    keras_params = model.count_params()
    tflite_size = len(tflite_model) / 1024  # KB
    
    # Keras accuracy
    keras_acc = accuracies[name] * 100
    
    # TFLite accuracy
    tflite_acc = evaluate_tflite_model(tflite_path, x_test, y_test) * 100
    
    summary_data.append({
        'Model': name,
        'Keras Params': f"{keras_params:,}",
        'TFLite Size (KB)': f"{tflite_size:.1f}",
        'Keras Accuracy (%)': f"{keras_acc:.2f}",
        'TFLite Accuracy (%)': f"{tflite_acc:.2f}"
    })

summary_df = pd.DataFrame(summary_data)
print("\n" + "="*80)
print("FINAL MODEL SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))

## 9. Dosyaları İndir

Google Colab'da çalışıyorsanız, aşağıdaki hücreyi çalıştırarak modelleri indirebilirsiniz:

In [ ]:
# Colab'da dosyaları zip olarak indir
try:
    from google.colab import files
    import shutil
    
    # models ve model_headers klasörlerini ziple
    shutil.make_archive('digit_recognition_models', 'zip', '.', 'models')
    shutil.make_archive('digit_recognition_headers', 'zip', '.', 'model_headers')
    
    print("Downloading model files...")
    files.download('digit_recognition_models.zip')
    files.download('digit_recognition_headers.zip')
    
except ImportError:
    print("Not running in Colab. Files saved to 'models/' and 'model_headers/' directories.")

## 10. ESP32-CAM Kullanım Talimatları

1. `model_headers/` klasöründeki istediğiniz model header dosyasını ESP32-CAM projenizin `include/` klasörüne kopyalayın
2. Ana kod dosyanızda header'ı include edin: `#include "squeezenet_model.h"` (örnek)
3. Model verisini TFLite Micro'ya yükleyin:
   ```cpp
   const tflite::Model* model = tflite::GetModel(squeezenet_model);
   ```
4. ESP32-CAM'den 28x28 gri tonlama görüntü alın ve modele input olarak verin

Detaylı ESP32-CAM kodu için `esp32_cam_inference/` projesine bakınız.